# ⚙️ Notebook 04 — Feature Engineering & Data Merging
**Owner:** Member 4

**Goal:** Create meaningful customer-level features from all datasets and merge them into one master table for modeling.

**Input:** Cleaned CSVs from `data/cleaned/`

**Output:** `data/merged/churn_master.csv`

---

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load cleaned datasets
renewal  = pd.read_csv('../data/cleaned/renewal_calls_clean.csv')
billing  = pd.read_csv('../data/cleaned/billings_clean.csv')
cc_calls = pd.read_csv('../data/cleaned/cc_calls_clean.csv')
emails   = pd.read_csv('../data/cleaned/emails_clean.csv')

print('All cleaned datasets loaded!')
print(f'Renewal: {renewal.shape}, Billing: {billing.shape}, CC: {cc_calls.shape}, Emails: {emails.shape}')

---
## Step 1: Create Features from Renewal Calls
This is the most important dataset — it contains the **churn label** (target variable).

In [ ]:
# Create the churn label: 1 if customer has any churn_category, 0 otherwise
renewal['is_churned'] = renewal['churn_category'].notna().astype(int)
print(f'Churn distribution:')
print(renewal.groupby('co_ref')['is_churned'].max().value_counts())

In [ ]:
# Aggregate renewal calls per customer
renewal_features = renewal.groupby('co_ref').agg(
    # Churn label (target) — 1 if churned in ANY call
    is_churned = ('is_churned', 'max'),
    
    # Call-related features
    total_renewal_calls = ('call_id', 'count'),
    
    # Complaint features
    serious_complaint_count = ('serious_complaint', lambda x: (x == 'Yes').sum()),
    other_complaint_count = ('other_complaint', lambda x: (x == 'Yes').sum()),
    
    # Competitor features
    competitor_mentioned = ('explicit_competitor_mention', lambda x: (x == 'Yes').any().astype(int)),
    switching_intent = ('explicit_switching_intent', lambda x: (x == 'Yes').any().astype(int)),
    
    # Price-related features
    price_increase_discussed = ('discussion_on_price_increase', lambda x: (x == 'Yes').any().astype(int)),
    discount_requested = ('discount_or_waiver_requested', lambda x: (x == 'Yes').any().astype(int)),
    discount_offered = ('discount_offered', lambda x: (x == 'Yes').any().astype(int)),
    
    # Cancel-related features
    desire_to_cancel = ('desire_to_cancel', lambda x: (x != 'Not Discussed').any().astype(int) if x.notna().any() else 0),
    
).reset_index()

print(f'Renewal features: {renewal_features.shape}')
renewal_features.head()

## Step 2: Create Features from Billings

> **NOTE:** You need to adjust column names below based on what the actual billing columns are (Member 1 & 2 will have documented this).

In [ ]:
# Check billing columns first
print('Billing columns:')
for col in billing.columns:
    print(f'  {col} — {billing[col].dtype} — {billing[col].nunique()} unique')

In [ ]:
# TODO: Aggregate billing features per customer
# Adjust column names based on what's actually in the dataset!
#
# billing_features = billing.groupby('co_ref').agg(
#     total_billed = ('amount_column', 'sum'),
#     avg_bill = ('amount_column', 'mean'),
#     billing_count = ('amount_column', 'count'),
#     # Add more features...
# ).reset_index()
#
# billing_features.head()

## Step 3: Create Features from CC Calls

In [ ]:
# Check CC calls columns
print('CC Calls columns:')
for col in cc_calls.columns:
    print(f'  {col} — {cc_calls[col].dtype} — {cc_calls[col].nunique()} unique')

In [ ]:
# TODO: Aggregate CC call features per customer
# cc_features = cc_calls.groupby('co_ref').agg(
#     cc_call_count = ('call_id_column', 'count'),
#     # Add more features...
# ).reset_index()
#
# cc_features.head()

## Step 4: Create Features from Emails

In [ ]:
# Check email columns
print('Email columns:')
for col in emails.columns:
    print(f'  {col} — {emails[col].dtype} — {emails[col].nunique()} unique')

In [ ]:
# TODO: Aggregate email features per customer
# email_features = emails.groupby('co_ref').agg(
#     email_count = ('email_id_column', 'count'),
#     # Add more features...
# ).reset_index()
#
# email_features.head()

---
## Step 5: Merge All Feature Tables

In [ ]:
# Start with renewal features (has the churn label)
master = renewal_features.copy()

# Merge billing features
# master = master.merge(billing_features, on='co_ref', how='left')

# Merge CC call features
# master = master.merge(cc_features, on='co_ref', how='left')

# Merge email features
# master = master.merge(email_features, on='co_ref', how='left')

print(f'Master dataset: {master.shape}')
master.head()

## Step 6: Post-Merge Cleanup

In [ ]:
# Fill NaN values after merge
# Customers who had no billing/cc/email records will have NaN — fill with 0
numeric_cols = master.select_dtypes(include=[np.number]).columns
master[numeric_cols] = master[numeric_cols].fillna(0)

print(f'Missing values after cleanup: {master.isnull().sum().sum()}')

## Step 7: Check Class Imbalance

In [ ]:
print('Target Variable Distribution:')
print(master['is_churned'].value_counts())
print(f'\nChurn Rate: {master["is_churned"].mean() * 100:.1f}%')

if master['is_churned'].mean() < 0.2 or master['is_churned'].mean() > 0.8:
    print('\n⚠️  WARNING: Class imbalance detected!')
    print('    Consider using SMOTE or class_weight="balanced" in models.')

## Step 8: Correlation Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))

# Only numeric columns
corr = master.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/11_correlation_heatmap.png', dpi=150)
plt.show()

## Step 9: Save Master Dataset

In [ ]:
# Save the final merged dataset
master.to_csv('../data/merged/churn_master.csv', index=False)

print('✅ Master dataset saved to data/merged/churn_master.csv')
print(f'   Shape: {master.shape[0]:,} customers × {master.shape[1]} features')
print(f'   Churn Rate: {master["is_churned"].mean()*100:.1f}%')
print(f'\nFeatures:')
for col in master.columns:
    print(f'   - {col}')